# AgenticPay OCL V2 — Constraint Bank 实验

这个 Notebook 会自动完成三次**相互独立**的真实 LLM 实验。每次实验都从空 Bank `L000` 开始，在训练 profile 上更新 Constraint Bank，再在独立验证集和测试集上评估。

每次运行都会保存完整 JSON artifact，Notebook 最后自动汇总规则晋升、拒绝原因、平均值和标准差。JSON 用于断点恢复和复核，不需要手工读取。

> 注意：默认会进行 3 次真实模型实验，会产生 API 调用和费用。运行前请确认 `OPENAI_API_KEY` 已设置。

In [ ]:
import json
import os
import statistics
import subprocess
import sys
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path


def locate_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'integrations/agenticpay_ocl_v2').is_dir() and (
            candidate / 'integrations/aocl_core'
        ).is_dir():
            return candidate
    raise RuntimeError('请从 amai_ocl 仓库内部启动 Jupyter。')


REPO_ROOT = locate_repo_root(Path.cwd())

# 实验配置
RUN_EXPERIMENTS = True
REPEATS = 3
MODEL = 'gpt-4o-mini'
TACTICS = ['privacy_phisher']
DERIVATION_LIMIT = 4
VALIDATION_LIMIT = 2
EVALUATION_LIMIT = 4
MAX_ROUNDS = 4
SKIP_ABLATION = True

OUTPUT_ROOT = REPO_ROOT / 'outputs/notebook_privacy_experiment'
SKILL_PATH = (
    REPO_ROOT
    / 'integrations/agenticpay_ocl_v2/data/candidate_instruction_skill.md'
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / '.env')
except ModuleNotFoundError:
    pass

experiment_env = os.environ.copy()
source_paths = [
    REPO_ROOT / 'integrations/aocl_core/src',
    REPO_ROOT / 'integrations/agenticpay_ocl_v2/src',
]
existing_pythonpath = experiment_env.get('PYTHONPATH')
pythonpath_parts = [str(path) for path in source_paths]
if existing_pythonpath:
    pythonpath_parts.append(existing_pythonpath)
experiment_env['PYTHONPATH'] = os.pathsep.join(pythonpath_parts)

print('Repository :', REPO_ROOT)
print('Output     :', OUTPUT_ROOT)
print('Repeats    :', REPEATS)
print('Model      :', MODEL)
print('Tactics    :', ', '.join(TACTICS))

In [ ]:
# 运行前检查：不会显示 API key 内容。
if not SKILL_PATH.is_file():
    raise FileNotFoundError(f'Candidate instruction skill 不存在：{SKILL_PATH}')

if RUN_EXPERIMENTS and not experiment_env.get('OPENAI_API_KEY'):
    raise RuntimeError('OPENAI_API_KEY 未设置。')

preflight = subprocess.run(
    [
        sys.executable,
        '-c',
        (
            'import agenticpay; '
            'import agenticpay_ocl_v2.batch_experiment; '
            "print('imports=ok')"
        ),
    ],
    cwd=REPO_ROOT,
    env=experiment_env,
    capture_output=True,
    text=True,
)
if preflight.returncode != 0:
    raise RuntimeError(preflight.stderr or preflight.stdout)

print(preflight.stdout.strip())
print('API key    : set' if experiment_env.get('OPENAI_API_KEY') else 'API key    : not needed')
print('Skill      :', SKILL_PATH.relative_to(REPO_ROOT))
print('Preflight complete.')

In [ ]:
def build_command():
    command = [
        sys.executable,
        '-u',
        '-m',
        'agenticpay_ocl_v2.batch_experiment',
        '--model',
        MODEL,
        '--tactics',
        *TACTICS,
        '--derivation-limit',
        str(DERIVATION_LIMIT),
        '--validation-limit',
        str(VALIDATION_LIMIT),
        '--evaluation-limit',
        str(EVALUATION_LIMIT),
        '--max-rounds',
        str(MAX_ROUNDS),
        '--candidate-instruction-skill',
        str(SKILL_PATH),
        '--output-root',
        str(OUTPUT_ROOT),
    ]
    if SKIP_ABLATION:
        command.append('--skip-ablation')
    return command


def complete_run_dirs():
    return {
        path.resolve()
        for path in OUTPUT_ROOT.glob('run-*')
        if (path / 'report.json').is_file()
    }


completed_run_dirs = []
log_root = OUTPUT_ROOT / 'notebook_logs'
log_root.mkdir(parents=True, exist_ok=True)

if RUN_EXPERIMENTS:
    for repeat in range(1, REPEATS + 1):
        print(f'\n========== RUN {repeat}/{REPEATS} ==========')
        before = complete_run_dirs()
        log_path = log_root / (
            f"invocation-{datetime.now().strftime('%Y%m%dT%H%M%S')}-{repeat}.log"
        )

        process = subprocess.Popen(
            build_command(),
            cwd=REPO_ROOT,
            env=experiment_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        suppress_large_report = False
        with log_path.open('w', encoding='utf-8') as log_file:
            assert process.stdout is not None
            for line in process.stdout:
                log_file.write(line)
                stripped = line.rstrip()
                if stripped.startswith('=== AgenticPay V2 library growth report'):
                    suppress_large_report = True
                    print('[run] final report written; suppressing large JSON output')
                elif (
                    not suppress_large_report
                    and stripped.startswith(
                        ('Artifacts:', '[run]', '[resume]', '[rerun]', 'Run stopped')
                    )
                ):
                    print(stripped)

        return_code = process.wait()
        after = complete_run_dirs()
        new_runs = sorted(after - before, key=lambda path: path.stat().st_mtime)

        if return_code not in (0, 2):
            raise RuntimeError(
                f'Run {repeat} 执行失败，返回码 {return_code}。查看 {log_path}'
            )
        if not new_runs:
            raise RuntimeError(
                f'Run {repeat} 没有生成完整 report.json。查看 {log_path}'
            )

        run_dir = new_runs[-1]
        completed_run_dirs.append(run_dir)
        report = json.loads((run_dir / 'report.json').read_text(encoding='utf-8'))
        final_row = report['growth_curve'][-1]
        status_counts = Counter(
            item.get('status', 'unknown')
            for item in report.get('learning_outcomes', [])
        )
        print('Run directory :', run_dir.relative_to(REPO_ROOT))
        print('Return code   :', return_code)
        print('Final version :', final_row['version_id'])
        print('Bank size     :', final_row['library_size'])
        print('Outcomes      :', dict(status_counts))
else:
    available = sorted(
        complete_run_dirs(), key=lambda path: path.stat().st_mtime
    )
    completed_run_dirs = available[-REPEATS:]
    print(f'RUN_EXPERIMENTS=False；加载最近 {len(completed_run_dirs)} 次完整实验。')

In [ ]:
# 读取三次 report，并生成每次运行的概要。
reports = []
summary_rows = []

for run_number, run_dir in enumerate(completed_run_dirs, start=1):
    report = json.loads((run_dir / 'report.json').read_text(encoding='utf-8'))
    reports.append(report)

    curve = report.get('growth_curve', [])
    baseline = curve[0]
    final = curve[-1]
    outcomes = report.get('learning_outcomes', [])
    counts = Counter(item.get('status', 'unknown') for item in outcomes)

    summary_rows.append(
        {
            'run': run_number,
            'directory': str(run_dir.relative_to(REPO_ROOT)),
            'baseline_version': baseline.get('version_id'),
            'final_version': final.get('version_id'),
            'bank_size': final.get('library_size', 0),
            'promoted': counts.get('promoted', 0),
            'rejected': counts.get('candidate_rejected', 0),
            'no_failure': counts.get('no_observed_failure', 0),
            'covered': counts.get('covered_by_library', 0),
            'intercept': final.get('attack_intercept_rate', 0.0),
            'false_block': final.get('benign_false_positive_rate', 0.0),
            'task_progress': final.get('task_progress_rate', 0.0),
            'valid_success': final.get('valid_success_rate', 0.0),
            'policy_failure': final.get('policy_failure_rate', 0.0),
            'executed_violations': final.get('executed_violation_steps', 0),
            'blocked_violations': final.get('blocked_violation_steps', 0),
            'mechanism_improved': report.get('mechanism_improved', False),
        }
    )

try:
    import pandas as pd
    from IPython.display import display

    display(pd.DataFrame(summary_rows))
except ModuleNotFoundError:
    for row in summary_rows:
        print(row)

In [ ]:
# 汇总 Baseline 与 Final 的平均值和标准差。
METRICS = {
    'intercept': 'attack_intercept_rate',
    'false_block': 'benign_false_positive_rate',
    'task_progress': 'task_progress_rate',
    'valid_success': 'valid_success_rate',
    'policy_failure': 'policy_failure_rate',
    'executed_violations': 'executed_violation_steps',
    'blocked_violations': 'blocked_violation_steps',
}


def mean_std(values):
    values = [float(value) for value in values]
    mean = statistics.mean(values)
    std = statistics.stdev(values) if len(values) > 1 else 0.0
    return mean, std


aggregate_rows = []
for stage, row_selector in (('L000 baseline', 0), ('Final bank', -1)):
    selected = [report['growth_curve'][row_selector] for report in reports]
    for display_name, field in METRICS.items():
        mean, std = mean_std(row.get(field, 0.0) for row in selected)
        aggregate_rows.append(
            {
                'stage': stage,
                'metric': display_name,
                'mean': mean,
                'std': std,
                'mean ± std': f'{mean:.3f} ± {std:.3f}',
            }
        )

try:
    display(pd.DataFrame(aggregate_rows))
except NameError:
    for row in aggregate_rows:
        print(f"{row['stage']:14s} {row['metric']:20s} {row['mean ± std']}")

In [ ]:
# 展开每次训练更新：这是判断“为什么没有学习”的关键记录。
outcome_rows = []

for run_number, report in enumerate(reports, start=1):
    for outcome in report.get('learning_outcomes', []):
        candidate = outcome.get('candidate') or {}
        outcome_rows.append(
            {
                'run': run_number,
                'profile': outcome.get('profile_id'),
                'status': outcome.get('status'),
                'version': outcome.get('version_id'),
                'candidate_id': outcome.get('candidate_id')
                or candidate.get('constraint_id'),
                'instruction': candidate.get('instruction', ''),
                'reasons': '; '.join(outcome.get('reasons', [])),
            }
        )

try:
    display(pd.DataFrame(outcome_rows))
except NameError:
    for row in outcome_rows:
        print(row)

print('\n状态解释：')
print('- promoted：候选规则通过验证并进入 Bank')
print('- candidate_rejected：生成了规则，但验证未通过')
print('- no_observed_failure：训练对话没有观察到 Seller 违规')
print('- covered_by_library：已有 Bank 已覆盖该失败，无需新增规则')

In [ ]:
# 可选图表：比较三次实验的 L000 与最终 Bank。
try:
    import matplotlib.pyplot as plt

    plot_metrics = [
        ('Policy failure', 'policy_failure_rate'),
        ('Attack intercept', 'attack_intercept_rate'),
        ('False block', 'benign_false_positive_rate'),
        ('Task progress', 'task_progress_rate'),
        ('Valid success', 'valid_success_rate'),
    ]
    labels = [item[0] for item in plot_metrics]
    baseline_means = [
        statistics.mean(report['growth_curve'][0].get(field, 0.0) for report in reports)
        for _, field in plot_metrics
    ]
    final_means = [
        statistics.mean(report['growth_curve'][-1].get(field, 0.0) for report in reports)
        for _, field in plot_metrics
    ]

    x = list(range(len(labels)))
    width = 0.36
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.bar([value - width / 2 for value in x], baseline_means, width, label='L000')
    ax.bar([value + width / 2 for value in x], final_means, width, label='Final Bank')
    ax.set_xticks(x, labels, rotation=20, ha='right')
    ax.set_ylim(0, 1)
    ax.set_ylabel('Mean rate across runs')
    ax.set_title(f'AgenticPay OCL V2 ({len(reports)} independent runs)')
    ax.legend()
    ax.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    plt.show()
except ModuleNotFoundError:
    print('matplotlib 未安装，跳过图表；数值分析不受影响。')

In [ ]:
# 保存 Notebook 汇总，便于之后直接复核而不重新调用 LLM。
combined_summary = {
    'created_at': datetime.now().isoformat(),
    'model': MODEL,
    'tactics': TACTICS,
    'runs': [str(path.relative_to(REPO_ROOT)) for path in completed_run_dirs],
    'run_summaries': summary_rows,
    'aggregate': aggregate_rows,
    'learning_outcomes': outcome_rows,
}
summary_path = OUTPUT_ROOT / 'notebook_summary.json'
summary_path.write_text(
    json.dumps(combined_summary, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print('Summary saved:', summary_path.relative_to(REPO_ROOT))